<a href="https://colab.research.google.com/github/anushah-200/factcheckAI/blob/main/notebooks/12_LOMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

BASE = "/content/drive/MyDrive/factcheckAI/outputs/"

Mounted at /content/drive


In [2]:
df = pd.read_csv(BASE + "factcheck_clean_dataset.csv")
print(df.shape)
df["Model"].value_counts()

(806, 19)


,count
Model,
OpenAI,279
DeepSeek,270
Groq,257


In [3]:
CAT_FEATURES = ["Category", "Type"]
NUM_FEATURES = ["ResponseLength", "QuestionLength", "ResponseCharacters", "AverageWordLength"]
TARGET = "Hallucination"

LOMO_EXPERIMENTS = [
    (["OpenAI", "Groq"], "DeepSeek"),
    (["OpenAI", "DeepSeek"], "Groq"),
    (["Groq", "DeepSeek"], "OpenAI"),
]

CLASSIFIERS = {
    "LogisticRegression": lambda: LogisticRegression(max_iter=1000, random_state=42),
    "DecisionTree": lambda: DecisionTreeClassifier(random_state=42),
    "RandomForest": lambda: RandomForestClassifier(n_estimators=200, random_state=42),
}

In [4]:
def build_features(train_df, test_df):
    train_df = train_df.copy()
    test_df = test_df.copy()

    for col in CAT_FEATURES:
        le = LabelEncoder()
        le.fit(train_df[col].astype(str))
        known = set(le.classes_)
        train_df[col + "_enc"] = le.transform(train_df[col].astype(str))
        test_df[col + "_enc"] = test_df[col].astype(str).apply(
            lambda x: le.transform([x])[0] if x in known else -1
        )

    scaler = StandardScaler()
    train_num = scaler.fit_transform(train_df[NUM_FEATURES])
    test_num = scaler.transform(test_df[NUM_FEATURES])

    train_cat = train_df[[c + "_enc" for c in CAT_FEATURES]].values
    test_cat = test_df[[c + "_enc" for c in CAT_FEATURES]].values

    X_train = np.hstack([train_cat, train_num])
    X_test = np.hstack([test_cat, test_num])

    return X_train, X_test, train_df[TARGET].values, test_df[TARGET].values

In [5]:
def evaluate(clf, X_test, y_test):
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_proba),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    }

In [6]:
all_results = []

for train_models, test_model in LOMO_EXPERIMENTS:
    train_df = df[df["Model"].isin(train_models)]
    test_df = df[df["Model"] == test_model]

    X_train, X_test, y_train, y_test = build_features(train_df, test_df)

    print(f"\n{'='*60}\nTrain {train_models} (n={len(train_df)}) -> Test {test_model} (n={len(test_df)})\n{'='*60}")

    for clf_name, clf_factory in CLASSIFIERS.items():
        clf = clf_factory()
        clf.fit(X_train, y_train)
        metrics = evaluate(clf, X_test, y_test)

        print(f"\n{clf_name}: Acc={metrics['Accuracy']:.4f}  F1={metrics['F1']:.4f}  ROC-AUC={metrics['ROC_AUC']:.4f}")

        all_results.append({
            "Train": "+".join(train_models), "Test": test_model,
            "Classifier": clf_name,
            **{k: v for k, v in metrics.items() if k != "confusion_matrix"},
        })


Train ['OpenAI', 'Groq'] (n=536) -> Test DeepSeek (n=270)

LogisticRegression: Acc=0.6074  F1=0.6159  ROC-AUC=0.6346

DecisionTree: Acc=0.7333  F1=0.7333  ROC-AUC=0.7392

RandomForest: Acc=0.7667  F1=0.7529  ROC-AUC=0.8186

Train ['OpenAI', 'DeepSeek'] (n=549) -> Test Groq (n=257)

LogisticRegression: Acc=0.6342  F1=0.7063  ROC-AUC=0.6962

DecisionTree: Acc=0.7938  F1=0.8239  ROC-AUC=0.7964

RandomForest: Acc=0.8560  F1=0.8810  ROC-AUC=0.9252

Train ['Groq', 'DeepSeek'] (n=527) -> Test OpenAI (n=279)

LogisticRegression: Acc=0.6416  F1=0.6988  ROC-AUC=0.6974

DecisionTree: Acc=0.7921  F1=0.8176  ROC-AUC=0.7880

RandomForest: Acc=0.8208  F1=0.8438  ROC-AUC=0.8952


In [7]:
for train_models, test_model in LOMO_EXPERIMENTS:
    train = df[df["Model"].isin(train_models)]
    test = df[df["Model"] == test_model]
    print(f"Train {train_models}: {train['Hallucination'].value_counts(normalize=True).round(3).to_dict()}")
    print(f"Test  {test_model}: {test['Hallucination'].value_counts(normalize=True).round(3).to_dict()}\n")

Train ['OpenAI', 'Groq']: {1: 0.591, 0: 0.409}
Test  DeepSeek: {0: 0.544, 1: 0.456}

Train ['OpenAI', 'DeepSeek']: {1: 0.514, 0: 0.486}
Test  Groq: {1: 0.615, 0: 0.385}

Train ['Groq', 'DeepSeek']: {1: 0.533, 0: 0.467}
Test  OpenAI: {1: 0.57, 0: 0.43}



In [8]:
results_df = pd.DataFrame(all_results)
results_df.to_csv(BASE + "lomo_results.csv", index=False)

rf_summary = results_df[results_df["Classifier"] == "RandomForest"][
    ["Train", "Test", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]
]
print(rf_summary.to_string(index=False))

          Train     Test  Accuracy  Precision   Recall       F1  ROC_AUC
    OpenAI+Groq DeepSeek  0.766667   0.727273 0.780488 0.752941 0.818566
OpenAI+DeepSeek     Groq  0.856031   0.895425 0.867089 0.881029 0.925233
  Groq+DeepSeek   OpenAI  0.820789   0.838509 0.849057 0.843750 0.895152
